# Project 2, Task 1: HR case (Data Analyst, Almaty): cleaning, funnel, scoring
**Input:** `data_raw.csv` (original HR export, 500 rows, two exports glued into one file).  
**Output:** `data_clean.csv` (one row per candidate, no names/phones/e-mails) and the two chosen candidates.  
Run top to bottom (`pandas`, `numpy`, `scipy` only). Every cleaning decision is a function below, so anyone gets the same file.

**Cleaning decisions that could have deleted the best candidate (and what I did instead)**
1. The file is two exports (comma format, then `;` format with a different column order) plus page-break lines, a footer and multi-line comments. Parsed by record, not by line.
2. Same ID, different person (`C1432`): both kept (second one gets suffix `-B`). Same person re-applying under a new ID (same name + birth date, new test score): keep the **latest** application, not the first.
3. `PostgreSQL` / `MySQL` count as SQL; `skills: SQL` written only in the HR comment counts too.
4. Missing / unclear values (`?`, `n/a`, `Intermediate`) are **not** treated as failures: they go to a re-check list.
5. City, university, age, gender, CV gap, `hired_before` are never used as filters.

In [1]:
import csv, io, re
SRC='data_raw.csv'   # the original export, untouched
ZW=re.compile('[\u200b\u200c\u200d\ufeff]')
def _records(lines, id_re):
    recs=[]; cur=None
    for l in lines:
        if id_re.match(ZW.sub('',l)):
            if cur is not None: recs.append(cur)
            cur=l
        elif cur is not None and cur.count('"')%2==1:   # open quoted field -> continuation
            cur+='\n'+l
        # else: blank line, page header, footer -> skip
    if cur is not None: recs.append(cur)
    return recs
def _split(rec, delim):
    return next(csv.reader(io.StringIO(rec),delimiter=delim,strict=False))
def load_raw():
    lines=open(SRC,encoding='utf-8').read().split('\n')
    i_end1=next(i for i,l in enumerate(lines) if l.startswith('ИТОГО по выгрузке 1'))
    i_imp2=next(i for i,l in enumerate(lines) if l.startswith('Import #2'))
    R1=_records(lines[3:i_end1], re.compile(r'^C\d+,'))
    R2=_records(lines[i_imp2+2:], re.compile(r'^[^;]*;C\d+;'))
    out1=[];out2=[]
    for r in R1:
        f=_split(r,',')
        if len(f)==22: f=f[:15]+[f[15]+','+f[16]]+f[17:]   # unquoted "690,000 KZT"
        out1.append(f)
    for r in R2:
        f=_split(r,';')
        out2.append(f)
    return out1,out2

In [2]:
"""Cleaning pipeline: raw HR export -> data_clean.csv  (run: python clean.py)"""
import re, datetime as dt
import numpy as np, pandas as pd

USD_KZT = 500          # assumption, see data note
EXPORT_DATE = dt.date(2026,10,29)

def fix_moji(s):
    """UTF-8 text that was mis-decoded as cp1251 ('РўР°СЂР°Р·' -> 'Тараз')."""
    if not isinstance(s,str): return s
    if re.search(r'[РС][\u0080-\u04ff]', s) and re.search(r'Р[ЎўЈ¤Ґ¦§Ё©Є«¬®Ї°±Ііґµ¶·ё№є»јЅѕїА-Яа-я]|С[ЂЃ‚ѓ„…†‡€‰Љ‹ЊЌЋЏђ‘’“”•–—™љ›њќћџ ЎўЈ¤Ґ¦§Ё©Є«¬®Ї]', s):
        try: return s.encode('cp1251').decode('utf-8')
        except Exception: return s
    return s

def fw(s):  # full-width digits -> ascii
    return s.translate({0xFF10+i:48+i for i in range(10)}) if isinstance(s,str) else s

MISSING = {'','?','-','n/a','na','null','none','нет','не указано','#ref!','#value!','#n/a','#div/0!','nan'}
def is_missing(s): return s is None or str(s).strip().lower() in MISSING

# ---------- experience (years) ----------
WORDS={'one':1,'two':2,'three':3,'four':4,'five':5,'six':6,'seven':7,'eight':8,'nine':9,'ten':10}
def parse_exp(s):
    s0=fw(str(s)).strip().lower()
    if s0 in ('no experience','нет опыта','стажер','0'): return 0.0,'ok'
    if s0 in MISSING: return np.nan,'missing'
    if s0 in WORDS: return float(WORDS[s0]),'ok'
    m=re.match(r'^(\d+(?:[.,]\d+)?)\s*\+?\s*(months?|мес\w*)?', s0)
    if m:
        v=float(m.group(1).replace(',','.'))
        if m.group(2): v/=12
        return v,'ok'
    return np.nan,'unparsed'

# ---------- skills ----------
def skill_flags(s):
    t=' '+str(s).lower().replace('powerbi','power bi')+' '
    sqlw=bool(re.search(r'\bsql\b|postgresql|mysql',t))
    ch='clickhouse' in t
    py=bool(re.search(r'python|пайтон|\bpy\b|pandas',t))
    bi=bool(re.search(r'power bi|tableau|looker',t))
    st=bool(re.search(r'\bstats\b|statistics|статистика|a/b|\bspss\b|(?<![a-z])r(?![a-z])',t))
    return sqlw,ch,py,bi,st

# ---------- English -> CEFR ----------
EN={'a2':'A2','basic':'A2','базовый':'A2','elementary':'A2',
    'intermediate':'B1*','средний':'B1*','b1':'B1',
    'upper-intermediate':'B2','upper int':'B2','выше среднего':'B2','b2':'B2','ielts 6.0':'B2',
    'advanced':'C1','c1':'C1','fluent':'C1','свободный':'C1','ielts 7.0':'C1',
    'c2':'C2','proficient':'C2','native-like':'C2','ielts 8.5':'C2'}
RANK={'A2':2,'B1':3,'B1*':3.5,'B2':4,'C1':5,'C2':6}   # B1* = "Intermediate": ambiguous B1/B2
def parse_en(s):
    k=str(s).strip().lower()
    if k in MISSING: return np.nan
    return EN.get(k,np.nan)

# ---------- scores ----------
def num(s):
    s=fw(str(s)).strip().lower().replace(',','.')
    return s
def parse_test(s):
    s=num(s)
    if s in MISSING: return np.nan,'missing'
    m=re.match(r'^(-?\d+(?:\.\d+)?)\s*(%|/100|/10)?$',s)
    if not m: return np.nan,'unparsed'
    v=float(m.group(1)); u=m.group(2)
    if u=='/10': v*=10
    if v<0 or v>100: return np.nan,'impossible'
    return v,'ok'
QUAL={'excellent':5,'good':4,'хорошо':4,'ok':3,'норм':3,'слабо':2,'weak':2,'poor':1}
def parse_int(s):
    raw=str(s).strip(); s=num(s)
    if s in MISSING: return np.nan,'missing'
    if s in QUAL: return float(QUAL[s]),'qualitative'
    m=re.match(r'^(\d+(?:\.\d+)?)\s*(%|/5)?$',s)
    if not m: return np.nan,'unparsed'
    v=float(m.group(1))
    if m.group(2)=='%': v=v/20
    if v<1 or v>5: return np.nan,'impossible'
    return v,'ok'

# ---------- salary (KZT / month) ----------
def parse_sal(s):
    s0=fw(str(s)).strip().lower()
    if s0 in MISSING or re.search(r'agree|договор|обсужд',s0): return np.nan,np.nan,'negotiable'
    s1=s0.replace('\xa0',' ')
    usd='$' in s1
    if 'млн' in s1:
        v=float(re.search(r'[\d.,]+',s1).group().replace(',','.'))*1e6; return v,v,'ok'
    nums=[x for x in re.findall(r'\d[\d ,.]*',s1)]
    vals=[]
    for n in nums:
        n=n.strip()
        n2=re.sub(r'[ ,]','',n) if re.search(r'\d[ ,]\d{3}\b',n) else n.replace(',','.').replace(' ','')
        try: vals.append(float(n2))
        except: pass
    if not vals: return np.nan,np.nan,'unparsed'
    if re.search(r'\d\s*k\b',s1): vals=[v*1000 for v in vals]
    if usd: vals=[v*USD_KZT for v in vals]
    lo,hi=min(vals),max(vals)
    return lo,hi,'ok'

# ---------- dates ----------
def parse_date(s, window=None, dob=False):
    s=fw(str(s)).strip()
    if s.lower() in MISSING: return pd.NaT,'missing'
    if re.fullmatch(r'\d{5}',s):
        d=dt.date(1899,12,30)+dt.timedelta(days=int(s)); return pd.Timestamp(d),'excel-serial'
    for f in ('%Y-%m-%d','%Y/%m/%d','%d.%m.%Y','%d %B %Y'):
        try: return pd.Timestamp(dt.datetime.strptime(s,f)),'ok'
        except: pass
    m=re.fullmatch(r'(\d{1,2})/(\d{1,2})/(\d{4})',s)
    if m:
        a,b,y=map(int,m.groups())
        if a>12: return pd.Timestamp(y,b,a),'ok'
        if b>12: return pd.Timestamp(y,a,b),'ok'
        if a==b: return pd.Timestamp(y,a,b),'ok'
        cand=[pd.Timestamp(y,b,a),pd.Timestamp(y,a,b)]          # dd/mm , mm/dd
        if window:
            inw=[c for c in cand if window[0]<=c<=window[1]]
            if len(inw)==1: return inw[0],'ok(resolved by application window)'
        return cand[0],'ambiguous(dd/mm assumed)'
    return pd.NaT,'unparsed'

def clean_id(s): return ZW.sub('',str(s)).strip().upper()

C1=['id','name','gender','dob','city','phone','email','university','degree','gpa','exp','skills','english','test','interview','salary','applied','source','gap','hired_before','comment']
C2=['name','id','email','phone','city','dob','gender','degree','university','gpa','english','exp','skills','interview','test','salary','source','applied','hired_before','gap','comment']

def load_unified():
    a,b=load_raw()
    d1=pd.DataFrame(a,columns=C1); d1['export']=1
    d2=pd.DataFrame(b,columns=C2)[C1]; d2['export']=2
    df=pd.concat([d1,d2],ignore_index=True)
    df['id']=df['id'].map(clean_id)
    return df

def gender(s):
    k=str(s).strip().lower()
    if k in ('male','m','м','муж'): return 'M'
    if k in ('female','f','ж','жен'): return 'F'
    return 'unknown'

def build():
    raw=load_unified()
    n_raw=len(raw)
    df=raw.copy()
    for c in ['name','city','university','degree','comment']: df[c]=df[c].map(fix_moji)
    # --- ID collisions: same ID but different person -> keep both, suffix the 2nd
    df['name_norm']=df.name.map(lambda x:' '.join(sorted(re.sub(r'[^\w\s]',' ',str(x).lower()).split())))
    seen={}
    for ix,r in df.iterrows():
        k=r['id']
        if k in seen and seen[k]!=r['name_norm']:
            df.at[ix,'id']=k+'-B'
        else: seen.setdefault(k,r['name_norm'])
    df=df.reset_index(drop=True)
    out=pd.DataFrame({'candidate_id':df.id})
    out['gender']=df.gender.map(gender)
    d=df.dob.map(lambda s: parse_date(s,dob=True))
    out['birth_date']=[x[0] for x in d]
    out['dob_note']=[x[1] for x in d]
    bad=(out.birth_date.dt.year>2008)|(out.birth_date.dt.year<1950)
    out.loc[bad,'dob_note']='impossible'; out.loc[bad,'birth_date']=pd.NaT
    out['age']=((pd.Timestamp(EXPORT_DATE)-out.birth_date).dt.days/365.25).round(0)
    # exp
    e=df.exp.map(parse_exp); out['exp_years']=[x[0] for x in e]; out['exp_note']=[x[1] for x in e]
    imp=(out.exp_years<0)|(out.exp_years>(out.age-17))|(out.exp_years>30)
    out.loc[imp,'exp_note']='impossible'; out.loc[imp,'exp_years']=np.nan
    # skills
    f=df.skills.map(skill_flags)
    out['skill_sql_literal']=[x[0] for x in f]; out['skill_clickhouse']=[x[1] for x in f]
    out['skill_python']=[x[2] for x in f]; out['skill_bi']=[x[3] for x in f]; out['skill_stats_ab']=[x[4] for x in f]
    # SQL recovered from HR comment ("skills: SQL, Python")
    out['sql_from_comment']=df.comment.str.contains(r'skills:.*sql',case=False,na=False)
    out['skill_sql']=out.skill_sql_literal|out.sql_from_comment
    out['english_cefr']=df.english.map(parse_en)
    out['english_rank']=out.english_cefr.map(RANK)
    t=df.test.map(parse_test); out['test_score']=[x[0] for x in t]; out['test_note']=[x[1] for x in t]
    i=df.interview.map(parse_int); out['interview_score']=[x[0] for x in i]; out['interview_note']=[x[1] for x in i]
    s=df.salary.map(parse_sal)
    out['salary_min_kzt']=[x[0] for x in s]; out['salary_max_kzt']=[x[1] for x in s]; out['salary_note']=[x[2] for x in s]
    out['salary_in_usd']=df.salary.str.contains(r'\$',regex=True).values
    sal_bad=(out.salary_min_kzt>5_000_000)|(out.salary_min_kzt<100_000)
    out.loc[sal_bad,['salary_min_kzt','salary_max_kzt']]=np.nan; out.loc[sal_bad,'salary_note']='impossible'
    WIN=(pd.Timestamp(2026,9,1),pd.Timestamp(2026,10,29))
    ap=df.applied.map(lambda x: parse_date(x,window=WIN)); out['applied_date']=[x[0] for x in ap]; out['applied_note']=[x[1] for x in ap]
    late=out.applied_date>pd.Timestamp(EXPORT_DATE)
    out.loc[late,'applied_note']='impossible(after export date)'; out.loc[late,'applied_date']=pd.NaT
    out['cv_gap_reason']=df.gap.replace('',np.nan)
    out['hired_before']=df.hired_before.str.lower().map(lambda x:{'yes':1,'да':1,'true':1,'1':1,'no':0,'нет':0,'false':0,'0':0}.get(x,np.nan))
    out['source_channel']=df.source.str.lower().replace({'рекомендация':'referral','сайт компании':'company site'})
    out['hr_comment']=df.comment.replace('',np.nan)
    out['export_block']=df.export
    out['_name_norm']=df.name_norm.values
    out['_filled']=df.apply(lambda r: sum(not is_missing(r[c]) for c in C1),axis=1).values
    # --- duplicates: (a) same ID + same name  (b) same normalised name + same birth date (re-application, new ID)
    out['_dob_key']=out.birth_date.dt.strftime('%Y-%m-%d').fillna('NA-'+out.candidate_id)
    out['_pk']=out._name_norm+'|'+out._dob_key
    out['_ad']=out.applied_date.fillna(pd.Timestamp('1900-01-01'))
    out=out.sort_values(['_pk','_ad','_filled'],ascending=[True,False,False])
    out['dup_of_rows']=out.groupby('_pk').candidate_id.transform('count')
    removed=out[out.duplicated('_pk',keep='first')]
    kept=out.drop_duplicates('_pk',keep='first')
    # same ID + same name but missing dob in all copies -> pk includes ID, so exact copies are caught there only if ids equal
    kept=kept.drop_duplicates('candidate_id',keep='first')
    kept['reapplied']=kept.dup_of_rows>1
    kept=kept.drop(columns=['_name_norm','_filled','_dob_key','_pk','_ad','dup_of_rows']).sort_values('candidate_id').reset_index(drop=True)
    kept.attrs['n_raw']=n_raw; kept.attrs['rows_removed']=n_raw-len(kept)
    kept.attrs['removed_ids']=removed.candidate_id.tolist()
    return kept, df

In [3]:
clean, work = build()
print('rows in raw file            :', clean.attrs['n_raw'])
print('unique candidates after dedup:', len(clean))
print('rows removed                :', clean.attrs['rows_removed'], clean.attrs['removed_ids'])
print('re-applicants (kept latest) :', int(clean.reapplied.sum()))
print()
for c in ['exp_note','test_note','interview_note','salary_note','applied_note']:
    print(f'{c:15s}', clean[c].value_counts(dropna=False).to_dict())
print()
print('English (CEFR, B1* = "Intermediate", ambiguous B1/B2):', clean.english_cefr.value_counts(dropna=False).to_dict())
print('SQL via comment only      :', int((clean.sql_from_comment & ~clean.skill_sql_literal).sum()))
print('Missing values per key column:')
print(clean[['exp_years','english_cefr','test_score','interview_score','salary_min_kzt']].isna().sum().to_string())

rows in raw file            : 500
unique candidates after dedup: 488
rows removed                : 12 ['C1360', 'C1346', 'C1226', 'C1475', 'C1389', 'C1295', 'C1413', 'C1427', 'C1393', 'C1229', 'C1320', 'C1385']
re-applicants (kept latest) : 12

exp_note        {'ok': 477, 'missing': 8, 'impossible': 2, 'unparsed': 1}
test_note       {'ok': 423, 'missing': 64, 'impossible': 1}
interview_note  {'ok': 352, 'missing': 76, 'qualitative': 59, 'impossible': 1}
salary_note     {'ok': 445, 'negotiable': 41, 'impossible': 2}
applied_note    {'ok': 420, 'ok(resolved by application window)': 60, 'ambiguous(dd/mm assumed)': 7, 'impossible(after export date)': 1}

English (CEFR, B1* = "Intermediate", ambiguous B1/B2): {'B2': 140, 'B1*': 116, 'C1': 93, 'A2': 61, 'B1': 35, nan: 23, 'C2': 20}
SQL via comment only      : 1
Missing values per key column:
exp_years          11
english_cefr       23
test_score         65
interview_score    77
salary_min_kzt     43


In [4]:
import numpy as np, pandas as pd
W={'test':0.30,'interview':0.25,'nice':0.30,'exp':0.15}
def components(o):
    c=pd.DataFrame(index=o.index)
    c['test']=o.test_score.clip(0,100)
    c['interview']=(o.interview_score-1)/4*100
    c['exp']=((o.exp_years.clip(upper=6)-2)/4*100).clip(lower=0)
    c['nice']=(o.skill_python.astype(int)+o.skill_bi.astype(int)+o.skill_stats_ab.astype(int)+(o.english_rank>=5).astype(int))/4*100
    return c
def score(o,w=W):
    c=components(o); return sum(c[k]*w[k] for k in w)
def gates(o,lenient=False):
    sql=o.skill_sql
    exp=o.exp_years>=2; en=o.english_rank>=4; sal=(o.salary_min_kzt<=700000)|o.salary_min_kzt.isna()
    te=o.test_score>=70; iv=o.interview_score>=4
    if lenient:
        exp|=o.exp_years.isna(); en|=o.english_rank.isna()|(o.english_cefr=='B1*'); te|=o.test_score.isna(); iv|=o.interview_score.isna()
    return sql,exp,en,sal,te,iv

In [5]:
# ---- FUNNEL: simple must-have rules, in order -------------------------------
o = clean.copy()
sql, exp, eng, sal, tst, itv = gates(o)
rules = [('SQL missing', sql), ('experience < 2 years or unknown', exp), ('English below B2 or unknown', eng),
         ('salary above 700,000 KZT (negotiable kept)', sal), ('company test < 70 or missing', tst), ('interview < 4 or missing', itv)]
rows = [('raw rows', clean.attrs['n_raw'], None), ('unique candidates (duplicates removed)', len(o), clean.attrs['n_raw']-len(o))]
mask = pd.Series(True, index=o.index)
for name, g in rules:
    before = int(mask.sum()); mask &= g; rows.append((name, int(mask.sum()), before-int(mask.sum())))
funnel = pd.DataFrame(rows, columns=['step', 'left', 'removed'])
print(funnel.to_string(index=False))
finalists = o[mask].copy()

                                      step  left  removed
                                  raw rows   500      NaN
    unique candidates (duplicates removed)   488     12.0
                               SQL missing   123    365.0
           experience < 2 years or unknown    83     40.0
               English below B2 or unknown    42     41.0
salary above 700,000 KZT (negotiable kept)    26     16.0
              company test < 70 or missing    10     16.0
                  interview < 4 or missing     5      5.0


In [6]:
# ---- WEIGHTED SCORE for the 5 who passed every must-have ----------------------
print('Weights:', W)
comp = components(finalists)
finalists['score'] = score(finalists)
show = finalists.join(comp.add_prefix('c_'))[['candidate_id','score','test_score','interview_score','exp_years','english_cefr','skill_python','skill_bi','skill_stats_ab','salary_min_kzt','cv_gap_reason','hr_comment']]
print(show.sort_values('score', ascending=False).round(1).to_string(index=False))

Weights: {'test': 0.3, 'interview': 0.25, 'nice': 0.3, 'exp': 0.15}
candidate_id  score  test_score  interview_score  exp_years english_cefr  skill_python  skill_bi  skill_stats_ab  salary_min_kzt     cv_gap_reason                       hr_comment
       C1131   95.4        97.0              5.0        5.0           C1          True      True            True        550000.0  декрет 2023-2024                              NaN
       C1133   80.7        94.0              4.0        3.0           C1          True      True            True        550000.0               NaN                              NaN
       C1097   75.4        93.0              5.0        8.0           C2         False     False           False             NaN учеба за границей см. письмо от 12.10\nperezvonit'
       C1220   60.6        77.0              4.0        3.0           B2         False      True            True        500000.0               NaN                              NaN
       C1172   56.0        74.0 

In [7]:
# ---- ROBUSTNESS 1: 5,000 random weight sets (Dirichlet) -----------------------
rng = np.random.default_rng(42); keys = list(W); cnt = pd.Series(0, index=finalists.index); N = 5000
for _ in range(N):
    w = dict(zip(keys, rng.dirichlet(np.ones(4)))); cnt[finalists.assign(s=score(finalists, w)).s.nlargest(2).index] += 1
print('share of random weight sets in which the candidate is in the top 2:')
print((cnt/N).rename(index=finalists.candidate_id).sort_values(ascending=False).round(3).to_string())

# ---- ROBUSTNESS 2: other cleaning choices --------------------------------------
def top2(frame):
    m = pd.Series(True, index=frame.index)
    for g in gates(frame): m &= g
    f = frame[m].copy(); f['score'] = score(f); return f.sort_values('score', ascending=False).candidate_id.head(2).tolist(), len(f)
print()
a = o.copy(); a['skill_sql'] = a.skill_sql | a.skill_clickhouse;                 print('ClickHouse also counts as SQL :', top2(a))
b = o.copy(); b['skill_sql'] = b.skill_sql_literal | b.sql_from_comment;         print('only literal "SQL" counts     :', top2(b))
for rate in (400, 500, 600):
    c = o.copy(); u = c.salary_in_usd
    c.loc[u, 'salary_min_kzt'] = c.loc[u, 'salary_min_kzt'] / 500 * rate
    print(f'USD/KZT = {rate}                :', top2(c))

share of random weight sets in which the candidate is in the top 2:
C1131    1.000
C1097    0.617
C1133    0.383
C1172    0.000
C1220    0.000

ClickHouse also counts as SQL : (['C1131', 'C1133'], 8)
only literal "SQL" counts     : (['C1131', 'C1133'], 5)
USD/KZT = 400                : (['C1131', 'C1133'], 5)
USD/KZT = 500                : (['C1131', 'C1133'], 5)
USD/KZT = 600                : (['C1131', 'C1133'], 5)


In [8]:
# ---- BLIND SPOT: candidates stopped only by MISSING or AMBIGUOUS data ---------
sl = gates(o, lenient=True); ml = pd.Series(True, index=o.index)
for g in sl: ml &= g
pool = o[ml & ~mask].copy()
best = pool.copy()
best['test_score'] = best.test_score.fillna(100); best['interview_score'] = best.interview_score.fillna(5)
best['exp_years'] = best.exp_years.fillna(6)
best['english_rank'] = np.where(best.english_cefr.eq('B1*') | best.english_rank.isna(), 6, best.english_rank)
pool['best_case_score'] = score(best)
pool['missing_or_unclear'] = pool.apply(lambda r: ', '.join([n for n, bad in [('test', pd.isna(r.test_score)), ('interview', pd.isna(r.interview_score)), ('experience', pd.isna(r.exp_years)), ('English', pd.isna(r.english_rank) or r.english_cefr == 'B1*')] if bad]), axis=1)
second = finalists.score.sort_values(ascending=False).iloc[1]
print(f'{len(pool)} people were kept alive only because a value is missing/unclear; to re-check list, not rejected.')
print('Score of my #2 (C1133):', round(second, 1), '| people whose BEST possible score beats it:', int((pool.best_case_score > second).sum()))
print(pool.sort_values('best_case_score', ascending=False)[['candidate_id','missing_or_unclear','best_case_score']].round(1).to_string(index=False))

13 people were kept alive only because a value is missing/unclear; to re-check list, not rejected.
Score of my #2 (C1133): 80.7 | people whose BEST possible score beats it: 3
candidate_id             missing_or_unclear  best_case_score
       C1117                           test             93.8
       C1305             interview, English             83.0
       C1271                        English             80.8
       C1042                test, interview             77.5
       C1296                        English             75.6
       C1003                      interview             75.6
       C1394                test, interview             73.8
       C1125                           test             71.2
       C1396                      interview             69.4
       C1132                      interview             68.8
       C1174 interview, experience, English             68.8
       C1350                        English             66.8
       C1245                    

In [9]:
# ---- FAIRNESS AUDIT (these columns were NOT used to filter or score) ----------
from scipy.stats import fisher_exact
a = o.copy(); a['pass_all'] = mask; a['has_cv_gap'] = a.cv_gap_reason.notna()
print('Gender of applicants:', a.gender.value_counts().to_dict())
print('Finalists by gender :', a[a.pass_all].gender.value_counts().to_dict())
fm = a[a.gender.isin(['F','M'])]
print()
for n, g in zip(['SQL','experience','English','salary','test','interview'], gates(o)):
    t = pd.crosstab(fm.gender, g[fm.index]); print(f'{n:11s} pass rate F={t.loc["F",True]/t.loc["F"].sum():.3f}  M={t.loc["M",True]/t.loc["M"].sum():.3f}  Fisher p={fisher_exact(t.values)[1]:.3f}')
t = pd.crosstab(a.has_cv_gap, a.pass_all)
print(); print('CV gap (maternity, army, illness...) vs passing all must-haves:'); print(t.to_string()); print('Fisher p =', round(fisher_exact(t.values)[1], 3))
print(); print('HR flags in comments (not skill criteria):')
print(o[o.hr_comment.str.contains('blacklist|offer|relative|директор|финал|ОТКАЗАЛСЯ', case=False, na=False)][['candidate_id','hr_comment']].to_string(index=False))
print(); print('Finalist comments:', finalists.set_index('candidate_id').hr_comment.to_dict())

Gender of applicants: {'M': 244, 'F': 221, 'unknown': 23}
Finalists by gender : {'F': 5}

SQL         pass rate F=0.235  M=0.266  Fisher p=0.456
experience  pass rate F=0.701  M=0.697  Fisher p=0.920
English     pass rate F=0.534  M=0.496  Fisher p=0.457
salary      pass rate F=0.742  M=0.680  Fisher p=0.153
test        pass rate F=0.353  M=0.238  Fisher p=0.008
interview   pass rate F=0.394  M=0.365  Fisher p=0.566

CV gap (maternity, army, illness...) vs passing all must-haves:
pass_all    False  True 
has_cv_gap              
False         211      3
True          272      2
Fisher p = 0.658

HR flags in comments (not skill criteria):
candidate_id                                                    hr_comment
       C1031                                                   offer sent?
       C1090                                                   offer sent?
       C1104                                           relative of manager
       C1121                                          

In [10]:
# ---- SAVE data_clean.csv (anonymised: no name / phone / e-mail / CV link) -----
final = clean.drop(columns=['birth_date', 'dob_note']).copy()
final['hr_comment'] = final.hr_comment.where(~final.hr_comment.str.startswith('http', na=False))   # drop CV links (personal data)
final['passes_all_must_haves'] = mask.values
final['weighted_score'] = np.nan
final.loc[finalists.index, 'weighted_score'] = finalists.score.round(2)
final['final_pick'] = final.candidate_id.isin(['C1131', 'C1133'])
final.to_csv('data_clean.csv', index=False)
print(final.shape); print(final[final.final_pick][['candidate_id','weighted_score','test_score','interview_score','exp_years','english_cefr']].to_string(index=False))

(488, 33)
candidate_id  weighted_score  test_score  interview_score  exp_years english_cefr
       C1131           95.35        97.0              5.0        5.0           C1
       C1133           80.70        94.0              4.0        3.0           C1


## Conclusion
* **Picks: C1131 and C1133.** C1131 leads in every scenario (top 2 in 100% of 5,000 random weight sets).
* **Second place is the sensitive part:** C1133 and C1097 swap depending on weights (C1097 is top 2 in about 62% of random sets, C1133 in about 38%). I chose C1133 because it meets all must-haves **and** all three analytics nice-to-haves (Python, Power BI, statistics) with a verified salary (about 550k KZT), while C1097 has none of them and a salary marked "by agreement". **C1097 is the reserve.**
* **Blind spot:** 13 people were kept off the shortlist only because a value is missing or unclear (not because they failed a rule). In the best case 3 of them (C1117, C1305, C1271) could beat my #2, and C1117 is the main one (missing test score): ask HR to collect those values before closing the search.
* Salary in USD was converted at 500 KZT/USD (assumption). Top 2 are unchanged at 400 and 600.